# 01 — Bronze : Ingestion brute

Charge les CSV bruts téléchargés par `scripts/download_data.py` et les écrit
en **Delta Lake** dans MinIO (S3-compatible), sans aucune transformation.

**Principe** : les données brutes sont préservées telles quelles.

In [ ]:
import os
import glob
import pandas as pd
from deltalake import write_deltalake, DeltaTable
from pathlib import Path

DATA_DIR = Path("../data")
BRONZE_PATH = str(DATA_DIR / "bronze" / "flights")

# MinIO / S3 config (si on utilise MinIO)
STORAGE_OPTIONS = {
    "AWS_ENDPOINT_URL": "http://localhost:9000",
    "AWS_ACCESS_KEY_ID": "minioadmin",
    "AWS_SECRET_ACCESS_KEY": "minioadmin123",
    "AWS_REGION": "us-east-1",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_ALLOW_HTTP": "true",
}

# On utilise le stockage local Delta pour simplifier
LOCAL_BRONZE = str(DATA_DIR / "bronze" / "flights")

In [ ]:
csv_files = sorted(DATA_DIR.glob("*.csv"))
print(f"Fichiers CSV trouvés : {len(csv_files)}")
for f in csv_files[:3]:
    print(f"  {f.name}")

In [ ]:
os.makedirs(LOCAL_BRONZE, exist_ok=True)

total_rows = 0
for i, csv_file in enumerate(csv_files):
    print(f"\n[{i+1}/{len(csv_files)}] {csv_file.name}")
    df = pd.read_csv(csv_file, low_memory=False, encoding="latin-1")
    print(f"  → {len(df):,} lignes, {len(df.columns)} colonnes")
    total_rows += len(df)

    write_deltalake(
        LOCAL_BRONZE,
        df,
        mode="append" if i > 0 else "overwrite",
    )
    print(f"  ✅ Écrit en Delta")

print(f"\n{'='*50}")
print(f"Total : {total_rows:,} lignes ingérées en bronze")

In [ ]:
dt = DeltaTable(LOCAL_BRONZE)
print(f"Delta Table version : {dt.version()}")
print(f"Fichiers Parquet : {len(dt.files())}")

df_check = dt.to_pandas()
print(f"Nombre total de lignes : {len(df_check):,}")
df_check.head(3)